In [1]:
import torch 
from torch import nn
import numpy


In [4]:
input_size =1
hidden_size = 32
lstm1 = nn.LSTM(input_size, hidden_size)

seq_len = 100
batch_size = 1
x = torch.randn(seq_len, batch_size, input_size)

output, (h_n, c_n) = lstm1(x)

assert output.shape == (seq_len, batch_size, hidden_size), "output shape mismatch"
assert h_n.shape == (1, batch_size, hidden_size), "hidden state shape mismatch"
assert c_n.shape == (1, batch_size, hidden_size), "cell state shape mismatch"

print(f"input shape: {x.shape}")
print(f"output shape: {output.shape}")
print(f"final hidden state shape: {h_n.shape}")
print(f"final cell state shape: {c_n.shape}")

input_size = 32
hidden_size = 1
lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size)#, proj_size=8)
seq_len = 100
batch_size = 1
x = output
output, (h_n,c_n) = lstm(output)

assert output.shape == (seq_len, batch_size, hidden_size), "output shape mismatch"
assert h_n.shape == (1, batch_size, hidden_size), "hidden state shape mismatch"
assert c_n.shape == (1, batch_size, hidden_size), "cell state shape mismatch"

print("2\n")
print(f"input shape: {x.shape}")
print(f"output shape: {output.shape}")
print(f"final hidden state shape: {h_n.shape}")
print(f"final cell state shape: {c_n.shape}")



input shape: torch.Size([100, 1, 1])
output shape: torch.Size([100, 1, 32])
final hidden state shape: torch.Size([1, 1, 32])
final cell state shape: torch.Size([1, 1, 32])
2

input shape: torch.Size([100, 1, 32])
output shape: torch.Size([100, 1, 1])
final hidden state shape: torch.Size([1, 1, 1])
final cell state shape: torch.Size([1, 1, 1])


In [5]:
## multi

num_layers = 3
lstm_multi = nn.LSTM(input_size, hidden_size, num_layers=num_layers)

x = torch.randn(seq_len, batch_size, input_size)
h_0 = torch.randn(num_layers, batch_size, hidden_size)
c_0 = torch.randn(num_layers, batch_size, hidden_size)

output, (h_n, c_n) = lstm_multi(x, (h_0,c_0))
assert output.shape == (seq_len, batch_size, hidden_size), "output shape mismatch"
assert h_n.shape == (num_layers, batch_size, hidden_size), "hidden state shape mismatch"
assert c_n.shape == (num_layers, batch_size, hidden_size), "cell state shape mismatch"

print(f"Multi layer lstm with {num_layers} layers:")
print(f"Output shape: {output.shape}, (last layer only)")
print(f"hidden states shape: {h_n.shape} (all layers)")
print(f"cell states shape: {c_n.shape}, (all layers)")


Multi layer lstm with 3 layers:
Output shape: torch.Size([100, 1, 32]), (last layer only)
hidden states shape: torch.Size([3, 1, 32]) (all layers)
cell states shape: torch.Size([3, 1, 32]), (all layers)


In [28]:
####moreno
from keras.layers import Input, Dense, LSTM, TimeDistributed, RepeatVector, Conv1D, \
    MaxPooling1D, UpSampling1D, Flatten, Reshape, GRU
from keras.models import Model
from keras import regularizers

def autoencoder_LSTM(X):
    inputs = Input(shape=(X.shape[1], X.shape[2]))
    print(inputs.shape)
    L1 = LSTM(32, activation='tanh', return_sequences=True, 
              kernel_regularizer=regularizers.l2(0.00))(inputs)
    print(L1.shape)
    L2 = LSTM(8, activation='tanh', return_sequences=False)(L1)
    print(L2.shape)
    L3 = RepeatVector(X.shape[1])(L2)
    print(L3.shape)
    L4 = LSTM(8, activation='tanh', return_sequences=True)(L3)
    L5 = LSTM(32, activation='tanh', return_sequences=True)(L4)
    output = TimeDistributed(Dense(X.shape[2]))(L5)    
    model = Model(inputs=inputs, outputs=output)
    return model

In [29]:
import numpy.random as nr
d0 = 1
d1 = 100
d2 = 1
a = nr.random((d0,d1,d2))
k = autoencoder_LSTM(a)

(None, 100, 1)
(None, 100, 32)
(None, 8)
(None, 100, 8)


In [12]:
import torch 
import torch.nn as nn

class AEric_Moreno(nn.Module):
    def __init__(self, sq_len, num_feat, exp_dim, compr_dim):
        super(AEric_moreno, self).__init__()

        self.sq_len = sq_len
        self.num_feat = num_feat
        self.exp_dim = exp_dim
        self.compr_dim = compr_dim

        self.EL1 = nn.LSTM(input_size=num_feat, 
                           hidden_size=exp_dim,
                            batch_first=True
                            )
        self.EL2 = nn.LSTM(input_size=exp_dim,
                            hidden_size=compr_dim, 
                            batch_first=True)
        
        self.DL1 = nn.LSTM(input_size=compr_dim, 
                           hidden_size=compr_dim,
                           batch_first=True)
        self.DL2 = nn.LSTM(input_size=compr_dim,
                           hidden_size=exp_dim,
                           batch_first=True)
    
    def forward(self, item):
        item, h_c = self.EL1(item)
        item, h_c = self.EL2(item)
        item = item[:,-1,:]
        item = item.repeat(1, self.sq_len, 1)
        item, h_c = self.DL1(item)
        item, h_c = self.DL2(item)




In [ ]:
net = AEric_Moreno(100,1, 32,8)

tensor([[ 0.8861, -1.0608]])
torch.Size([1, 2])
tensor([[[ 0.8861, -1.0608],
         [ 0.8861, -1.0608],
         [ 0.8861, -1.0608],
         [ 0.8861, -1.0608],
         [ 0.8861, -1.0608]]])
torch.Size([1, 5, 2])
tensor([[[ 0.8861, -1.0608]]])
torch.Size([1, 1, 2])
tensor([[[ 0.8861, -1.0608],
         [ 0.8861, -1.0608],
         [ 0.8861, -1.0608],
         [ 0.8861, -1.0608],
         [ 0.8861, -1.0608]]])
torch.Size([1, 5, 2])
